# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [6]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [3]:
print('datasize:', df.shape)
print('thông tin dữ liệu:')
df.info()


datasize: (541909, 8)
thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


## A.2. Missing values & Duplicate data

In [4]:
print('số lượng giá trị bị thiếu:\n', df.isnull().sum())
print('số lượng giá trị bị trùng:\n', df.duplicated().sum())

số lượng giá trị bị thiếu:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
số lượng giá trị bị trùng:
 5268


## A.3. Invalid values

In [ ]:
display(df.describe())



,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [7]:
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df['Sales'] = df['Quantity'] * df['UnitPrice']

print(df[['Quantity', 'UnitPrice', 'Sales']].head())


   Quantity  UnitPrice  Sales
0         6       2.55  15.30
1         6       3.39  20.34
2         8       2.75  22.00
3         6       3.39  20.34
4         6       3.39  20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [ ]:
numeric_cols = ['Quantity', 'UnitPrice', 'Sales']
print("MEAN (Trung bình)")
print(df[numeric_cols].mean())
print("\nMEDIAN (Trung vị)")
print(df[numeric_cols].median())
print("\nMODE (Yếu vị)")
print(df.mode().iloc[0])


--- MEAN (Trung bình) ---
Quantity     10.542037
UnitPrice     3.907625
Sales        20.121871
dtype: float64

--- MEDIAN (Trung vị) ---
Quantity     3.00
UnitPrice    2.08
Sales        9.90
dtype: float64

--- MODE (Yếu vị) ---
InvoiceNo                                  573585
StockCode                                  85123A
Description    WHITE HANGING HEART T-LIGHT HOLDER
Quantity                                        1
InvoiceDate                   2011-10-31 14:41:00
UnitPrice                                    1.25
CustomerID                                17841.0
Country                            United Kingdom
Sales                                        15.0
Name: 0, dtype: object


## Group 2 — Dispersion

In [9]:
print("STANDARD DEVIATION (Độ lệch chuẩn)")
print(df[numeric_cols].std())
print("\nMIN & MAX")
print(df[numeric_cols].agg(['min', 'max']))


STANDARD DEVIATION (Độ lệch chuẩn)
Quantity     155.524124
UnitPrice     35.915681
Sales        270.356743
dtype: float64

MIN & MAX
     Quantity  UnitPrice       Sales
min         1      0.001       0.001
max     80995  13541.330  168469.600


## Group 3 — Location and Shape

In [10]:
print("SKEWNESS (Độ lệch)")
print(df[numeric_cols].skew())
print("\nKURTOSIS (Độ nhọn)")
print(df[numeric_cols].kurt())


SKEWNESS (Độ lệch)
Quantity     471.727716
UnitPrice    206.087555
Sales        506.706012
dtype: float64

KURTOSIS (Độ nhọn)
Quantity     236462.342826
UnitPrice     62483.142715
Sales        297651.661046
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [14]:
sales_by_country = df.groupby('Country')['Sales'].sum().sort_values(ascending=False)
top_country = sales_by_country.index[0]
top_sales = sales_by_country.iloc[0]
total_sales = sales_by_country.sum()
phantram = (top_sales / total_sales) * 100

print(f"-> {top_country} đóng góp cao nhất: {phantram:.2f}% tổng doanh thu.")


-> United Kingdom đóng góp cao nhất: 84.61% tổng doanh thu.


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [15]:
sales_by_product = df.groupby(['StockCode', 'Description'])['Sales'].sum().sort_values(ascending=False)
print("Top 5 sản phẩm mang lại doanh thu cao nhất:")
print(sales_by_product.head())


Top 5 sản phẩm mang lại doanh thu cao nhất:
StockCode  Description                       
DOT        DOTCOM POSTAGE                        206248.77
22423      REGENCY CAKESTAND 3 TIER              174484.74
23843      PAPER CRAFT , LITTLE BIRDIE           168469.60
85123A     WHITE HANGING HEART T-LIGHT HOLDER    104340.29
47566      PARTY BUNTING                          99504.33
Name: Sales, dtype: float64


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [18]:
df['Month'] = df['InvoiceDate'].dt.month
df.groupby('Month')['Sales'].sum()




Month
1      691364.560
2      523631.890
3      717639.360
4      537808.621
5      770536.020
6      761739.900
7      719221.191
8      759138.380
9     1058590.172
10    1154979.300
11    1509496.330
12    1462538.820
Name: Sales, dtype: float64

Doanh thu bắt đầu tăng mạnh từ tháng 9 và đạt đỉnh điểm vào tháng 11. Sang tháng 12 doanh thu sụt giảm mạnh

## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [19]:

order_values = df.groupby(['Country', 'InvoiceNo'])['Sales'].sum().reset_index()
aov_by_country = order_values.groupby('Country')['Sales'].mean().sort_values(ascending=False)

print("Top 10 quốc gia có Giá trị đơn hàng trung bình cao nhất:")
print(aov_by_country.head(10))

Top 10 quốc gia có Giá trị đơn hàng trung bình cao nhất:
Country
Singapore      3039.898571
Netherlands    3036.663191
Australia      2430.198421
Japan          1969.282632
Lebanon        1693.880000
Hong Kong      1426.527273
Brazil         1143.600000
Sweden         1066.064722
Switzerland    1057.220370
Denmark        1053.074444
Name: Sales, dtype: float64


## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [ ]:
df_raw = pd.read_csv(csv_path)


total_orders = df_raw.groupby('Country')['InvoiceNo'].nunique()
cancel_orders = df_raw[df_raw['Quantity'] < 0].groupby('Country')['InvoiceNo'].nunique()


cancel_rate = (cancel_orders / total_orders * 100).dropna().sort_values(ascending=False)
print("Top 10 quốc gia có tỷ lệ trả hàng/hủy đơn cao nhất (%):")
print(cancel_rate.head(10))

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Doanh nghiệp phụ thuộc rất lớn vào thị trường nội địa (Vương quốc Anh - UK), chiếm tỷ trọng doanh thu áp đảo so với phần còn lại của thế giới.

Hoạt động kinh doanh mang tính mùa vụ rõ rệt, bùng nổ vào quý cuối năm (tháng 9 đến tháng 11).

Phân khúc khách hàng có sự khác biệt về hành vi mua sắm: Khách hàng tại UK mua thường xuyên nhưng giá trị mỗi đơn nhỏ (bán lẻ), trong khi khách hàng quốc tế (Hà Lan, Úc, Nhật) mua ít lần hơn nhưng giá trị đơn hàng cực kỳ lớn (bán buôn).

Tỷ lệ hoàn hủy đơn hàng có sự chênh lệch lớn giữa các quốc gia